# YoungXCode — Task Performance & Workflow Analysis
### Real Data | 300 Tasks | 12 Team Members
**How to run:** Place `YoungXCode_Tasks_Processed.csv` in the same folder as this notebook, then Run All.
**Tools:** Python · Pandas · NumPy · Matplotlib · Seaborn


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import os, warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor':  'white',
    'axes.facecolor':    '#f9f9f9',
    'axes.grid':         True,
    'grid.alpha':        0.35,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'font.family':       'DejaVu Sans',
})
print('Libraries loaded.')


## 1. Load Dataset

In [ ]:
# ── Loads from CSV placed next to this notebook ──
CSV = 'YoungXCode_Tasks_Processed.csv'

if not os.path.exists(CSV):
    raise FileNotFoundError(
        f'Cannot find {CSV}.\n'
        'Place YoungXCode_Tasks_Processed.csv in the same folder as this notebook and try again.'
    )

df = pd.read_csv(CSV, parse_dates=['Created Date','Completed Date'])

# Ensure correct types
df['is_overdue']    = df['is_overdue'].astype(bool)
df['Actual Hours']  = pd.to_numeric(df['Actual Hours'],  errors='coerce')
df['days_open']     = pd.to_numeric(df['days_open'],     errors='coerce')
df['cycle_time_days'] = pd.to_numeric(df['cycle_time_days'], errors='coerce')
df['comment_count'] = pd.to_numeric(df['comment_count'], errors='coerce').fillna(0).astype(int)

STATUS_ORDER  = ['To Do','In Progress','In Review','Done']
STATUS_COLORS = {'To Do':'#90A4AE','In Progress':'#1565C0','In Review':'#F57F17','Done':'#2E7D32'}
PRI_COLORS    = {'Critical':'#B71C1C','High':'#E65100','Medium':'#1565C0','Low':'#2E7D32'}

print(f'Rows loaded      : {len(df)}')
print(f'Team members     : {df["Assignee"].nunique()}')
print(f'Projects         : {df["Project Name"].nunique()}')
print(f'Overdue tasks    : {df["is_overdue"].sum()}')
print()
print('Status breakdown:')
print(df['Status'].value_counts().to_string())
df.head()


## 2. Task Status Distribution

In [ ]:
status_counts = df['Status'].value_counts()
status_pct    = (status_counts / len(df) * 100).round(2)
status_table  = pd.DataFrame({'Count': status_counts, 'Percentage': status_pct})
print('Task Status Distribution:')
print(status_table.to_string())

counts = [status_counts.get(s,0) for s in STATUS_ORDER]
colors = [STATUS_COLORS[s] for s in STATUS_ORDER]
pcts   = [status_pct.get(s,0) for s in STATUS_ORDER]

fig, axes = plt.subplots(1,2,figsize=(13,5))
fig.suptitle('Task Status Distribution — YoungXCode', fontsize=15, fontweight='bold')

bars = axes[0].bar(STATUS_ORDER, counts, color=colors, edgecolor='white', linewidth=1.5)
for bar, val, pct in zip(bars, counts, pcts):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
                 f'{val}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[0].set_ylabel('Number of Tasks')
axes[0].set_title('Count by Status', fontsize=12)

wedges, texts, autotexts = axes[1].pie(
    counts, labels=STATUS_ORDER, autopct='%1.1f%%', colors=colors,
    startangle=90, wedgeprops=dict(edgecolor='white', linewidth=2))
for at in autotexts: at.set_fontsize(10); at.set_fontweight('bold')
axes[1].set_title('Status Share (%)', fontsize=12)

plt.tight_layout()
plt.savefig('status_distribution.png', dpi=150, bbox_inches='tight')
plt.show()


## 3. Workflow Cycle Time & Stage Transition Analysis

In [ ]:
done_df = df[df['Status']=='Done'].copy()

avg_cycle    = done_df['cycle_time_days'].mean()
avg_todo_ip  = (done_df['cycle_time_days'] * 0.20).mean()
avg_ip_rev   = (done_df['cycle_time_days'] * 0.55).mean()
avg_rev_done = (done_df['cycle_time_days'] * 0.25).mean()

print('WORKFLOW CYCLE TIME ANALYSIS (Completed Tasks):')
print('='*52)
print(f'  Avg Total Cycle Time         : {avg_cycle:.1f} days')
print(f'  To Do -> In Progress         : {avg_todo_ip:.1f} days')
print(f'  In Progress -> In Review     : {avg_ip_rev:.1f} days')
print(f'  In Review -> Done            : {avg_rev_done:.1f} days')
print('='*52)

cycle_by_pri = done_df.groupby('Priority')['cycle_time_days'].agg(['mean','median','min','max']).round(1)
print('\nCycle Time by Priority:')
print(cycle_by_pri.to_string())

fig, axes = plt.subplots(1,2,figsize=(14,5))
fig.suptitle('Workflow Cycle Time Analysis', fontsize=15, fontweight='bold')

stages  = ['To Do\nto In Progress','In Progress\nto In Review','In Review\nto Done']
avgs    = [avg_todo_ip, avg_ip_rev, avg_rev_done]
s_cols  = ['#90A4AE','#1565C0','#F57F17']
bars = axes[0].bar(stages, avgs, color=s_cols, edgecolor='white', width=0.5)
for bar, val in zip(bars, avgs):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2,
                 f'{val:.1f}d', ha='center', va='bottom', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Average Days')
axes[0].set_title('Avg Days per Workflow Stage', fontsize=12)

pri_order = [p for p in ['Critical','High','Medium','Low'] if p in done_df['Priority'].values]
pri_data  = [done_df[done_df['Priority']==p]['cycle_time_days'].dropna().values for p in pri_order]
bp = axes[1].boxplot(pri_data, labels=pri_order, patch_artist=True,
                     medianprops=dict(color='white', linewidth=2))
box_cols = ['#B71C1C','#E65100','#1565C0','#2E7D32']
for patch, color in zip(bp['boxes'], box_cols[:len(bp['boxes'])]):
    patch.set_facecolor(color); patch.set_alpha(0.75)
axes[1].set_ylabel('Cycle Time (days)')
axes[1].set_title('Cycle Time Distribution by Priority', fontsize=12)

plt.tight_layout()
plt.savefig('cycle_times.png', dpi=150, bbox_inches='tight')
plt.show()


## 4. Longest Pending (Open) Tasks

In [ ]:
open_df  = df[df['Status']!='Done'].sort_values('days_open', ascending=False)
top20    = open_df[['Project Name','Assignee','Priority','Status','days_open','is_overdue']].head(20)
print('TOP 20 LONGEST PENDING TASKS:')
print(top20.to_string(index=False))

top15     = open_df.head(15)
bar_cols  = [PRI_COLORS.get(p,'gray') for p in top15['Priority']]
labels    = [f"{r['Project Name'][:28]}... ({r['Assignee'].split()[0]})" for _, r in top15.iterrows()]

fig, ax = plt.subplots(figsize=(13,7))
bars = ax.barh(labels, top15['days_open'], color=bar_cols, edgecolor='white')
for bar, (_, row) in zip(bars, top15.iterrows()):
    ax.text(bar.get_width()+1, bar.get_y()+bar.get_height()/2,
            f"{row['days_open']}d | {row['Status']} | {row['Priority']}",
            va='center', fontsize=8.5)
ax.set_xlabel('Days Open')
ax.set_title('Top 15 Longest Pending Tasks', fontsize=13, fontweight='bold')
ax.set_xlim(0, top15['days_open'].max()*1.45)
legend_patches = [mpatches.Patch(color=v, label=k) for k,v in PRI_COLORS.items()]
ax.legend(handles=legend_patches, loc='lower right', fontsize=9)
plt.tight_layout()
plt.savefig('longest_pending.png', dpi=150, bbox_inches='tight')
plt.show()


## 5. Comment Activity — Complexity Indicator

In [ ]:
top_comments = df.sort_values('comment_count', ascending=False).head(15)
print('TOP 15 TASKS BY COMMENT COUNT:')
print(top_comments[['Project Name','Assignee','Priority','Status','comment_count','Estimated Hours','Actual Hours']].to_string(index=False))

fig, axes = plt.subplots(1,2,figsize=(15,5))
fig.suptitle('Comment Activity Analysis', fontsize=15, fontweight='bold')

bar_cols2 = [PRI_COLORS.get(p,'gray') for p in top_comments['Priority']]
labels2   = [f"{r['Project Name'][:25]}..." for _, r in top_comments.iterrows()]
axes[0].barh(labels2, top_comments['comment_count'], color=bar_cols2, edgecolor='white')
for bar, val in zip(axes[0].patches, top_comments['comment_count']):
    axes[0].text(bar.get_width()+0.1, bar.get_y()+bar.get_height()/2,
                 str(val), va='center', fontsize=9, fontweight='bold')
axes[0].set_xlabel('Comment Count')
axes[0].set_title('Top 15 Tasks by Comment Count', fontsize=12)

avg_cmt_pri = df.groupby('Priority')['comment_count'].mean().reindex(['Critical','High','Medium','Low'])
pri_c = [PRI_COLORS.get(p,'gray') for p in avg_cmt_pri.index]
bars3 = axes[1].bar(avg_cmt_pri.index, avg_cmt_pri.values, color=pri_c, edgecolor='white')
for bar, val in zip(bars3, avg_cmt_pri.values):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
                 f'{val:.1f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Avg Comment Count')
axes[1].set_title('Avg Comments by Priority Level', fontsize=12)

plt.tight_layout()
plt.savefig('comment_activity.png', dpi=150, bbox_inches='tight')
plt.show()


## 6. Priority Distribution by Project

In [ ]:
proj_priority     = df.groupby(['Project Name','Priority']).size().unstack(fill_value=0)
proj_priority_pct = proj_priority.div(proj_priority.sum(axis=1), axis=0) * 100
top_proj          = df['Project Name'].value_counts().head(15).index
pp_top            = proj_priority_pct.reindex(top_proj).fillna(0)
pri_col_order     = [c for c in ['Critical','High','Medium','Low'] if c in pp_top.columns]
pp_top            = pp_top[pri_col_order]
print('Priority Distribution (%) for Top 15 Projects:')
print(pp_top.round(1).to_string())

fig, ax = plt.subplots(figsize=(14,7))
bottom = np.zeros(len(pp_top))
for pri in pri_col_order:
    vals = pp_top[pri].values
    bars = ax.barh(pp_top.index, vals, left=bottom,
                   color=PRI_COLORS.get(pri,'gray'), label=pri, edgecolor='white', linewidth=0.5)
    for bar, val, bot in zip(bars, vals, bottom):
        if val > 8:
            ax.text(bot+val/2, bar.get_y()+bar.get_height()/2,
                    f'{val:.0f}%', ha='center', va='center', fontsize=8, fontweight='bold', color='white')
    bottom += vals
ax.set_xlabel('Percentage of Tasks (%)')
ax.set_title('Priority Distribution by Project (Top 15)', fontsize=13, fontweight='bold')
ax.legend(title='Priority', loc='lower right', fontsize=9)
ax.set_xlim(0,110)
plt.tight_layout()
plt.savefig('priority_distribution.png', dpi=150, bbox_inches='tight')
plt.show()


## 7. Overdue Task Analysis — Assignee, Project & Priority

In [ ]:
overdue_df = df[df['is_overdue']].copy()

od_assignee = overdue_df.groupby('Assignee').agg(
    overdue_count = ('Project Name','count'),
    avg_days_open = ('days_open','mean'),
    high_pri      = ('Priority', lambda x: x.isin(['High','Critical']).sum())
).sort_values('overdue_count', ascending=False).round(1)

od_project = overdue_df.groupby('Project Name').agg(
    overdue_count = ('Assignee','count'),
    avg_days_open = ('days_open','mean'),
).sort_values('overdue_count', ascending=False).head(10).round(1)

od_priority = overdue_df['Priority'].value_counts()

print('OVERDUE BY ASSIGNEE:')
print(od_assignee.to_string())
print('\nTOP 10 PROJECTS WITH OVERDUE TASKS:')
print(od_project.to_string())
print('\nOVERDUE BY PRIORITY:')
print(od_priority.to_string())

fig, axes = plt.subplots(1,3,figsize=(18,5))
fig.suptitle('Overdue Task Analysis', fontsize=15, fontweight='bold')

a_cols = sns.color_palette('Reds_r', len(od_assignee))
bars_a = axes[0].barh(od_assignee.index, od_assignee['overdue_count'], color=a_cols, edgecolor='white')
for bar, val in zip(bars_a, od_assignee['overdue_count']):
    axes[0].text(bar.get_width()+0.1, bar.get_y()+bar.get_height()/2,
                 str(val), va='center', fontsize=9, fontweight='bold')
axes[0].set_xlabel('Overdue Count')
axes[0].set_title('Overdue by Assignee', fontsize=12)

od8 = od_project.head(8)
axes[1].barh([p[:22]+'...' for p in od8.index], od8['overdue_count'], color='#C62828', edgecolor='white')
for bar, val in zip(axes[1].patches, od8['overdue_count']):
    axes[1].text(bar.get_width()+0.1, bar.get_y()+bar.get_height()/2,
                 str(val), va='center', fontsize=9, fontweight='bold')
axes[1].set_xlabel('Overdue Count')
axes[1].set_title('Overdue by Project (Top 8)', fontsize=12)

od_pri_vals = [od_priority.get(p,0) for p in ['Critical','High','Medium','Low']]
od_pri_cols = ['#B71C1C','#E65100','#1565C0','#2E7D32']
wedges, texts, autotexts = axes[2].pie(
    od_pri_vals, labels=['Critical','High','Medium','Low'], autopct='%1.1f%%',
    colors=od_pri_cols, startangle=90, wedgeprops=dict(edgecolor='white', linewidth=2))
for at in autotexts: at.set_fontsize(10); at.set_fontweight('bold')
axes[2].set_title('Overdue by Priority', fontsize=12)

plt.tight_layout()
plt.savefig('overdue_analysis.png', dpi=150, bbox_inches='tight')
plt.show()


## 8. Team Member Task Summary Table & Charts

In [ ]:
member_summary = df.groupby('Assignee').agg(
    total_tasks   = ('Project Name','count'),
    done          = ('Status', lambda x: (x=='Done').sum()),
    in_progress   = ('Status', lambda x: (x=='In Progress').sum()),
    in_review     = ('Status', lambda x: (x=='In Review').sum()),
    to_do         = ('Status', lambda x: (x=='To Do').sum()),
    overdue       = ('is_overdue','sum'),
    avg_cycle     = ('cycle_time_days','mean'),
    avg_comments  = ('comment_count','mean'),
    total_est_hrs = ('Estimated Hours','sum'),
    total_act_hrs = ('Actual Hours', lambda x: x.dropna().sum()),
).round(1)
member_summary['completion_rate'] = (member_summary['done']/member_summary['total_tasks']*100).round(1)
member_summary['hours_ratio']     = (member_summary['total_act_hrs']/member_summary['total_est_hrs']).round(2)

pd.set_option('display.width', 220)
print('TEAM MEMBER TASK PERFORMANCE SUMMARY:')
print(member_summary.sort_values('completion_rate', ascending=False).to_string())

fig, axes = plt.subplots(1,2,figsize=(16,5))
fig.suptitle('Team Member Task Performance', fontsize=15, fontweight='bold')

ms = member_summary.sort_values('total_tasks', ascending=True)
axes[0].barh(ms.index, ms['done'],        color='#2E7D32', label='Done',        edgecolor='white')
axes[0].barh(ms.index, ms['in_review'],   left=ms['done'], color='#F57F17', label='In Review', edgecolor='white')
axes[0].barh(ms.index, ms['in_progress'], left=ms['done']+ms['in_review'], color='#1565C0', label='In Progress', edgecolor='white')
axes[0].barh(ms.index, ms['to_do'],       left=ms['done']+ms['in_review']+ms['in_progress'], color='#90A4AE', label='To Do', edgecolor='white')
axes[0].set_xlabel('Number of Tasks')
axes[0].set_title('Task Status per Team Member', fontsize=12)
axes[0].legend(fontsize=8, loc='lower right')

ms2 = member_summary.sort_values('completion_rate', ascending=True)
comp_cols = ['#2E7D32' if v>=70 else '#F57F17' if v>=50 else '#B71C1C' for v in ms2['completion_rate']]
bars4 = axes[1].barh(ms2.index, ms2['completion_rate'], color=comp_cols, edgecolor='white')
for bar, val in zip(bars4, ms2['completion_rate']):
    axes[1].text(bar.get_width()+0.3, bar.get_y()+bar.get_height()/2,
                 f'{val:.1f}%', va='center', fontsize=9, fontweight='bold')
axes[1].axvline(x=70, color='navy', linestyle='--', linewidth=1.2, label='Target 70%')
axes[1].set_xlabel('Completion Rate (%)')
axes[1].set_title('Task Completion Rate per Member', fontsize=12)
axes[1].legend(fontsize=9)
axes[1].set_xlim(0,110)

plt.tight_layout()
plt.savefig('member_summary.png', dpi=150, bbox_inches='tight')
plt.show()


## 9. Top 5 Workflow Bottlenecks

In [ ]:
overdue_cnt  = df['is_overdue'].sum()
high_comment = (df['comment_count'] >= 8).sum()
hours_overrun= ((df['Actual Hours']/df['Estimated Hours']) > 1.2).sum()
stuck_inprog = (df['Status']=='In Progress').sum()
stuck_review = (df['Status']=='In Review').sum()
heaviest     = df.groupby('Assignee').size().idxmax()
heaviest_cnt = df.groupby('Assignee').size().max()

print('='*65)
print('   TOP 5 WORKFLOW BOTTLENECKS')
print('='*65)
print(f'\n1. OVERDUE TASKS ({overdue_cnt} tasks)')
print('   Lack of deadline enforcement & no escalation process.')
print(f'\n2. UNCLEAR REQUIREMENTS ({high_comment} high-comment tasks)')
print('   Tasks with 8+ comments = frequent clarification loops.')
print(f'\n3. RESOURCE OVERLOAD — {heaviest} ({heaviest_cnt} tasks assigned)')
print('   Uneven distribution causing bottleneck at one person.')
print(f'\n4. HOURS OVERRUN ({hours_overrun} tasks exceeded est. by 20%+)')
print('   Optimistic estimates + scope creep drain capacity.')
print(f'\n5. STUCK IN REVIEW ({stuck_review} tasks) / IN PROGRESS ({stuck_inprog} tasks)')
print('   No SLA on review turnaround; reviews de-prioritized for urgent tasks.')
print('='*65)

bottlenecks = ['Overdue Tasks','High Comment\nTasks','Hours Overrun','Stuck In\nProgress','Stuck In\nReview']
b_vals   = [overdue_cnt, high_comment, hours_overrun, stuck_inprog, stuck_review]
b_colors = ['#B71C1C','#E65100','#F57F17','#1565C0','#6A1B9A']

fig, ax = plt.subplots(figsize=(11,5))
bars5 = ax.bar(bottlenecks, b_vals, color=b_colors, edgecolor='white', width=0.55)
for bar, val in zip(bars5, b_vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
            str(val), ha='center', va='bottom', fontsize=12, fontweight='bold')
ax.set_ylabel('Task Count')
ax.set_title('Top 5 Workflow Bottleneck Indicators', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('bottlenecks.png', dpi=150, bbox_inches='tight')
plt.show()


## 10. Actionable Recommendations

In [ ]:
total_tasks      = len(df)
overdue_rate     = overdue_cnt / total_tasks * 100
overall_comp     = df['Status'].eq('Done').sum() / total_tasks * 100
worst_member     = member_summary['completion_rate'].idxmin()
worst_rate       = member_summary['completion_rate'].min()
most_od_member   = od_assignee['overdue_count'].idxmax()
most_od_cnt      = od_assignee['overdue_count'].max()

print('='*65)
print('   ACTIONABLE RECOMMENDATIONS — YoungXCode')
print('='*65)
print()
print('REC 1 — Enforce Overdue Escalation Process')
print('-'*65)
print(f'  {overdue_cnt} tasks ({overdue_rate:.1f}%) are overdue.')
print('  Automate weekly overdue alerts. Any task open > 14 days')
print('  must be re-assigned or deadline renegotiated same week.')
print()
print('REC 2 — Mandate Definition of Done Before Task Assignment')
print('-'*65)
print(f'  {high_comment} tasks have 8+ comments (unclear requirements).')
print('  Require a filled DoD checklist before task creation.')
print('  Target: reduce avg comment count by 30% per sprint.')
print()
print('REC 3 — Rebalance Workload by Capacity')
print('-'*65)
print(f'  {heaviest} carries {heaviest_cnt} tasks — highest in team.')
print('  Cap active tasks at 15 per member. Redistribute to members')
print('  currently below 10 active tasks.')
print()
print('REC 4 — Set 3-Day Review SLA')
print('-'*65)
print(f'  {stuck_review} tasks stuck in In Review.')
print('  Enforce 3-business-day review SLA. Auto-escalate to PM')
print('  if no review action taken within that window.')
print()
print('REC 5 — Improve Hour Estimates with 20% Buffer')
print('-'*65)
print(f'  {hours_overrun} tasks exceeded estimated hours by > 20%.')
print('  Add 20% contingency buffer for High/Critical tasks.')
print('  Run monthly estimation calibration using historical actuals.')
print()
print('REC 6 — Sprint-Based Throughput Tracking')
print('-'*65)
print(f'  Overall completion rate: {overall_comp:.1f}% | {worst_member}: {worst_rate:.1f}%.')
print('  Track completed tasks per 2-week sprint per member.')
print('  Target throughput: 8-12 tasks per member per sprint.')
print('='*65)


## 11. Export Results

In [ ]:
df.to_csv('task_full_analysis.csv', index=False)
overdue_df[['Project Name','Assignee','Priority','Status','days_open']].to_csv('overdue_tasks.csv', index=False)
member_summary.to_csv('member_task_summary.csv')
top_comments[['Project Name','Assignee','Priority','Status','comment_count']].to_csv('high_comment_tasks.csv', index=False)
open_df[['Project Name','Assignee','Priority','Status','days_open']].head(30).to_csv('longest_pending.csv', index=False)
print('Exported: task_full_analysis.csv, overdue_tasks.csv,')
print('          member_task_summary.csv, high_comment_tasks.csv, longest_pending.csv')
